In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextAttention
MODEL = "google/gemma-4-E4B-it" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [3]:
model = load_model()

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [4]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

Storing kv for 22 and 23 sliding and full layers, layers after use them, dont calculate.

In [7]:
model.model.language_model.layers[0].self_attn.config.num_kv_shared_layers

18

In [8]:
model.model.language_model.layers[0].self_attn.config.num_hidden_layers


42

In [9]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [10]:
for i in range(42):
    print(i, bridge.blocks[i].attn.is_kv_shared_layer,
          bridge.blocks[i].attn.store_full_length_kv,
          bridge.blocks[0].attn._original_component.config.layer_types[i])

0 False False sliding_attention
1 False False sliding_attention
2 False False sliding_attention
3 False False sliding_attention
4 False False sliding_attention
5 False False full_attention
6 False False sliding_attention
7 False False sliding_attention
8 False False sliding_attention
9 False False sliding_attention
10 False False sliding_attention
11 False False full_attention
12 False False sliding_attention
13 False False sliding_attention
14 False False sliding_attention
15 False False sliding_attention
16 False False sliding_attention
17 False False full_attention
18 False False sliding_attention
19 False False sliding_attention
20 False False sliding_attention
21 False False sliding_attention
22 False True sliding_attention
23 False True full_attention
24 True False sliding_attention
25 True False sliding_attention
26 True False sliding_attention
27 True False sliding_attention
28 True False sliding_attention
29 True False full_attention
30 True False sliding_attention
31 True Fal

In [14]:
bridge


TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [15]:
model.model.language_model.layers[0].self_attn.is_kv_shared_layer

False

In [16]:
model.model.language_model.layers[0].self_attn.kv_shared_layer_index

In [17]:
model.model.language_model.layers[0].self_attn.store_full_length_kv

False

In [18]:
model.model.language_model.layers[0].self_attn.use_alternative_attention

False

In [19]:
model.model.language_model.layers[0].self_attn.o_proj._original_component.bias

In [20]:
model.model.language_model.layers[0].self_attn._original_component.config.attention_bias

False

In [21]:
model.model.language_model.layers[0].self_attn._original_component.config._attn_implementation

'sdpa'

In [22]:
model.model.language_model.layers[41].self_attn._original_component.config.use_double_wide_mlp

False

In [23]:
model.model.language_model.layers[0]._original_component.enable_moe_block

False

In [24]:
model.model.language_model.layers[0]._original_component.hidden_size_per_layer_input

256

основной поток 2560 ───────────────────────────────┐
                                                   + → 2560
основной поток 2560 → gate 256                     │
                              × per-layer input 256 │
                              → projection 2560 ────┘

There is another branch of per_layer_input in forward of model.

Builded from special per-layer embeddings. Or from projection of main embedding (???)

In [25]:
# token t:
    # main embedding/hidden state: 2560
    # layer 0 extra input:          256
    # layer 1 extra input:          256
    # ...
    # layer 41 extra input:         256

In [26]:
model.model.language_model.layers[0]._original_component.hidden_size

2560

In [27]:
model.model.language_model.layers[0]._original_component.config.hidden_activation

'gelu_pytorch_tanh'

In [28]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [29]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [30]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [31]:
ignored_params = []
# for name, param in model.named_parameters():
#     if "norm" in name:
#         ignored_params.append(param)

In [32]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=True,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
        unwrapped_parameters=[(bridge.blocks[0].attn.q_norm._original_component.weight, 0),]
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.language_model.layers.13._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.37._original_component.self_attn._original_component.k_norm._original_component.weight', 'model.audio_tower.layers.3.feed_forward2.pre_layer_norm.weight', 'model.language_model.layers.39._original_component.post_per_layer_input_norm._original_component.weight', 'model.vision_tower._original_component.encoder.layers.5.self_attn.q_norm.weight', 'model.vision_tower._original_component.encoder.layers.13.post_feedforward_layernorm.weight', 'model.audio_tower.layers.3.lconv1d.conv_norm.weight', 'model.audio_tower.layers.9.lconv1d.linear_end.linear.weight', 'model.audio_tower.layers.11.feed_forward2.pre_layer_norm.weight', 'model.embed_audio.embedding_projection.weight', 'model.language_model.layers.25._orig

In [33]:
bridge.get_submodule("blocks.0.attn.q.hook_out")

HookPoint(name='blocks.0.attn.q.hook_out')

In [34]:
#name of module, cols (in), rows(out)


#local configuration
d = {"blocks.0.attn.q": (None, [2, 6, 9]), #repeat indices for qkvo
     "blocks.0.mlp.up_proj": (None, [1, 3, 5])}

In [35]:
group = DG.get_pruning_group(
    bridge.blocks[5].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.5._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)) => prune_out_channels on blocks.5._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.5._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)) => prune_out_channels on _Reshape_2439(), len(idxs)=3
[2] prune_out_channels on _Reshape_2439() => prune_out_channels on _ElementWiseOp_2434(ToCopyBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_2434(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_2432(MulBackward0), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_2434(ToCopyBackward0) => prune_out_chann

In [ ]:
group = DG.get_pruning_group(
    bridge.blocks[22].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )
print(group)
print("found all dependent modules of kv cache")


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.22._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on blocks.22._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.22._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on _Reshape_3229(), len(idxs)=3
[2] prune_out_channels on _Reshape_3229() => prune_out_channels on _ElementWiseOp_3224(ToCopyBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_3224(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_3222(MulBackward0), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_3224(ToCopyBackward0) => prune_out_ch

In [48]:
group = DG.get_pruning_group(
    bridge.blocks[23].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.23._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)) => prune_out_channels on blocks.23._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.23._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=4096, bias=False)) => prune_out_channels on _Reshape_5039(), len(idxs)=3
[2] prune_out_channels on _Reshape_5039() => prune_out_channels on _ElementWiseOp_5034(ToCopyBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_5034(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_5032(MulBackward0), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_5034(ToCopyBackward0) => prune_out_ch

In [ ]:
# BUG: dependency goes on sdpa to another layers, but for full attn it cant!
#so cache works only partially, full attn cache is broken.

# In group it becomes:  
#     [22] _Reshape_5009 → BmmBackward0  
#     [25] ... → AddBackward0  
#     [26] ... → SafeSoftmaxBackward0  
#     [29] ... → BmmBackward0  

# scores = Q @ K.transpose(-2, -1)

# Q:   [B, H, S_q, D]
# Kᵀ:  [B, H, D, S_k]
# out: [B, H, S_q, S_k]

# that the inner dimension D, so TP can't correctly link it as dependency dimension

# why it works for sdpa?
# for sdpa:
# it shows as ScaledDotProductFlashAttentionBackward0 in deps for layer 22, so dependency goes in general way
# and fanout other layers through it.